In [ ]:
!pip install transformers accelerate peft datasets bitsandbytes trl

In [ ]:
#upload the datasets
from google.colab import files
uploaded = files.upload()

Saving dataset_clean.csv to dataset_clean (2).csv
Saving training_data_for_ft.json to training_data_for_ft (2).json


In [ ]:
#Load training data
import json
import pandas as pd
from datasets import Dataset

# Load training data
with open("training_data_for_ft.json") as f:
    nb_data = json.load(f)

# The notebook cell content is stored as a list of strings.
raw = ''.join(nb_data['cells'][0]['source'])
# This transforms the string into a list of dictionaries.
training_data = json.loads(raw)


# Format data for fine-tuning
def format_example(ex):
    return f"Frage: {ex['prompt']}\nAntwort: {ex['answer']}"
# Here we loop over the full dataset and convert every question-answer pair into the instruction format above.
texts = [format_example(ex) for ex in training_data]

#we create a HuggingFace Dataset with one column(text)
dataset = Dataset.from_dict({"text": texts})


Training examples: 100
Sample:
 <|im_start|>user
Was ist die steuerliche Bemessungsgrundlage für die Körperschaftsteuer?
<|im_end|>
<|im_start|>assistant
Die Bemessungsgrundlage ist das steuerpflichtige Einkommen der Körperschaft nach Verlustausgleich und zulässigen Abzügen gemäß KStG § 7.
<|im_end|>


In [ ]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

#This clears unused Python objects from memory, I used it because I got out-of-memory errors
gc.collect()
torch.cuda.empty_cache()

MODEL_ID = "tiiuae/falcon-rw-1b"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
# I use left padding so the real prompt stays at the end

# 4-bit instead of 16/32, because it reduces GPU memory usage
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto")

model = prepare_model_for_kbit_training(model)

# LoRA configuration
# LoRA adds small trainable matrices to selected layers instead of updating the full model
lora_config = LoraConfig(
    r=16, # rank of LoRA matrices
    lora_alpha=32, # scaling factor
    target_modules=["query_key_value"], # Falcon attention projection
    lora_dropout=0.05, # dropout for regularization
    bias="none", # no bias fine-tuning
    task_type=TaskType.CAUSAL_LM)

# Only a very small subset of parameters will now be updated
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 3,145,728 || all params: 1,417,793,536 || trainable%: 0.2219


In [ ]:
def tokenize(examples):
    tokenized = tokenizer(
        examples["text"],
        padding="max_length", # pad all sequences to the same fixed length
        truncation=True, # cut off text if it is longer than max_length
        max_length=512 # maximum sequence length used for training
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized
# Apply tokenization to the full dataset in batches for faster preprocessing
tokenized_dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
from transformers import Trainer, TrainingArguments

# Define the main training settings
training_args = TrainingArguments(
    output_dir="./falcon-tax-lora", # folder where the fine-tuned LoRA adapter will be saved
    per_device_train_batch_size=1, # process 1 example at a time on the GPU
    gradient_accumulation_steps=4,
    learning_rate=2e-4, # step size for updating LoRA weights
    num_train_epochs=4, # train for 4 full passes over the dataset
    fp16=True, # use half precision to reduce GPU memory usage
    logging_steps=10, # print training progress every 10 steps
    save_strategy="no",
    dataloader_pin_memory=False
)
# Create the Hugging Face Trainer object
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)
# fine-tuning process
trainer.train()

model.save_pretrained("./falcon-tax-lora")
tokenizer.save_pretrained("./falcon-tax-lora")


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,4.957914
20,0.632350
30,0.435651
40,0.333392
50,0.288296
60,0.266790
70,0.258854
80,0.237331
90,0.238125
100,0.222219


Training done!


In [ ]:
import gc, torch

gc.collect()
torch.cuda.empty_cache()
model.eval()

# Load test dataset with questions
test_df = pd.read_csv("dataset_clean.csv")
print(f"Test questions: {len(test_df)}")

# Function to generate an answer for a single question
def generate_answer(question):
    prompt = f"Frage: {question}\nAntwort:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# Disable gradient computation for faster and memory-efficient inference
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    # Remove the input prompt from the output tokens
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    # Decode only newly generated tokens into text
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# Run inference over the full test dataset
answers = []
for i, row in test_df.iterrows():
    answers.append(generate_answer(row['prompt']))
    #we print progress after every 50 examples
    if (i + 1) % 50 == 0 or (i + 1) == len(test_df):
        print(f"Done: {i+1}/{len(test_df)}")



The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Test questions: 643
Done: 50/643
Done: 100/643
Done: 150/643
Done: 200/643
Done: 250/643
Done: 300/643
Done: 350/643
Done: 400/643
Done: 450/643
Done: 500/643
Done: 550/643
Done: 600/643
Done: 643/643
Inference done!


In [ ]:
results_df = pd.DataFrame({"id": test_df["id"], "answer": answers})
results_df.to_csv("ft_results.csv", index=False)


from google.colab import files
files.download("ft_results.csv")

             id                                             answer
0  CORP-TAX-001  Die grundsätzige Verfügung steuerliche Einkünf...
1  CORP-TAX-002                                               Eine
2  CORP-TAX-003  Gewerbebetrieben sind verpflichtet, sämtliche ...
3  CORP-TAX-004                                                  E
4  CORP-TAX-005  Ja, die Verfügungen aus einer Verfügung steuer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>